In [8]:
import torch
import torchvision
from torchvision import datasets, transforms, models
from torch import nn, optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import os

print("PyTorch version:", torch.__version__)
print("GPU tersedia:", torch.cuda.is_available())

PyTorch version: 2.12.0+cpu
GPU tersedia: False


In [9]:
# Sesuaikan path ke folder data/raw kamu
data_dir = os.path.abspath("../data/raw/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)")


In [10]:
# Transformasi gambar — resize, augmentasi, ubah ke tensor
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),       # Samakan ukuran semua foto
    transforms.RandomHorizontalFlip(),   # Flip acak (supaya AI lebih pintar)
    transforms.RandomRotation(10),       # Rotasi acak sedikit
    transforms.ToTensor(),               # Ubah gambar jadi angka
    transforms.Normalize([0.485, 0.456, 0.406],  # Normalisasi warna
                         [0.229, 0.224, 0.225])
])

valid_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Load dataset dari folder
train_data = datasets.ImageFolder(os.path.join(data_dir, "train"), transform=train_transforms)
valid_data = datasets.ImageFolder(os.path.join(data_dir, "valid"), transform=valid_transforms)

# DataLoader — pengumpan foto ke AI saat training
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=32, shuffle=False)

print("Jumlah foto training :", len(train_data))
print("Jumlah foto validasi :", len(valid_data))
print("Jumlah kelas penyakit:", len(train_data.classes))
print("\nContoh nama kelas:")
for c in train_data.classes[:5]:
    print(" -", c)

Jumlah foto training : 70295
Jumlah foto validasi : 17572
Jumlah kelas penyakit: 38

Contoh nama kelas:
 - Apple___Apple_scab
 - Apple___Black_rot
 - Apple___Cedar_apple_rust
 - Apple___healthy
 - Blueberry___healthy


In [11]:
# Load ResNet18 yang sudah pre-trained (sudah pintar kenali gambar umum)
model = models.resnet18(weights="IMAGENET1K_V1")

# Bekukan semua layer lama — kita hanya latih ulang bagian terakhir
for param in model.parameters():
    param.requires_grad = False

# Ganti layer terakhir sesuai jumlah penyakit tanaman kita
jumlah_kelas = len(train_data.classes)
model.fc = nn.Linear(model.fc.in_features, jumlah_kelas)

# Pakai GPU kalau ada, kalau tidak pakai CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Model siap! Jumlah kelas:", jumlah_kelas)
print("Training di:", device)

Model siap! Jumlah kelas: 38
Training di: cpu


In [12]:
# Fungsi loss dan optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# Training loop
EPOCHS = 5  # Mulai dari 5 dulu, bisa ditambah nanti
train_losses, valid_losses, valid_accs = [], [], []

for epoch in range(EPOCHS):
    # === TRAINING ===
    model.train()
    running_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        output = model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # === VALIDASI ===
    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            output = model(images)
            val_loss += criterion(output, labels).item()
            _, predicted = torch.max(output, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    acc = correct / total * 100
    train_losses.append(running_loss / len(train_loader))
    valid_losses.append(val_loss / len(valid_loader))
    valid_accs.append(acc)

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Train Loss: {running_loss/len(train_loader):.3f} | "
          f"Val Loss: {val_loss/len(valid_loader):.3f} | "
          f"Akurasi: {acc:.1f}%")

print("\nTraining selesai!")

KeyboardInterrupt: 

In [ ]:
# Simpan model ke folder model/
os.makedirs("model", exist_ok=True)
torch.save({
    'model_state_dict': model.state_dict(),
    'class_names': train_data.classes,
}, "../model/plant_model.pth")
print("Model tersimpan di: ../model/plant_model.pth")

# Tampilkan grafik akurasi
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label="Train Loss")
plt.plot(valid_losses, label="Val Loss")
plt.title("Loss per Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(valid_accs, color="green", label="Akurasi")
plt.title("Akurasi Validasi per Epoch")
plt.legend()
plt.tight_layout()
plt.show()